# 02 — Text Chunking

**Vai trò:** Data Engineer · **Task:** S2-DE-05 (Yêu cầu 2, 9.3)

Notebook này khám phá `TextChunker` (S2-DE-01/02) — bước **Chunk** thứ hai trong luồng `Load → Chunk → Embed → Store`. Ta sẽ so sánh ba chiến lược (`FIXED_SIZE`, `RECURSIVE`, `SEMANTIC`), quan sát ảnh hưởng của `chunk_size`/`chunk_overlap`, và xác minh trực tiếp **Property 1** (chunk bao phủ trọn vẹn nội dung gốc) cùng **Property 2** (`chunk.doc_id == document.doc_id`).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker
from src.models import ChunkStrategy

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\lh222k\AI-Research-Assistant-with-RAG


## 1. Tải tài liệu mẫu

Tái sử dụng tài liệu mẫu đã tạo ở [`01_document_loading.ipynb`](01_document_loading.ipynb) (`data/raw/sample_rag_overview.txt`). Nếu chưa chạy notebook đó, cell dưới sẽ tạo lại file mẫu để notebook này vẫn chạy độc lập được (Yêu cầu 9.1 — notebook phải chạy từ đầu đến cuối không lỗi).

In [2]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
sample_path = RAW_DIR / "sample_rag_overview.txt"

if not sample_path.exists():
    sample_path.write_text(
        "Retrieval-Augmented Generation (RAG) la kien truc ket hop "
        "retrieval va generation. He thong tim cac doan van ban lien "
        "quan tu kho du lieu rieng truoc khi yeu cau LLM sinh cau tra "
        "loi, giup giam hien tuong ao giac (hallucination).\n\n"
        "Quy trinh RAG gom hai giai doan: indexing (tai, chia nho, tao "
        "embedding, luu vao vector store) va querying (embed cau hoi, "
        "tim chunk gan nhat, ghep prompt, goi LLM). Moi vai tro ky su "
        "phu trach mot phan: Data Engineer lo tai lieu va vector, "
        "Pipeline Engineer dieu phoi RAGPipeline va prompt, Model "
        "Engineer ket noi OLLAMA cho embedding va sinh van ban.\n",
        encoding="utf-8",
    )
    print(f"Đã tạo lại: {sample_path.relative_to(PROJECT_ROOT)}")

loader = DocumentLoader()
document = loader.load(str(sample_path))
print(f"doc_id  = {document.doc_id}")
print(f"độ dài  = {len(document.content)} ký tự")

doc_id  = 4c69f82b6454088f
độ dài  = 1255 ký tự


## 2. So sánh ba chiến lược chunking

`ChunkStrategy` có ba giá trị: `FIXED_SIZE` (cửa sổ trượt theo số ký tự cố định), `RECURSIVE` (ưu tiên ranh giới đoạn văn → câu → từ), và `SEMANTIC` (theo ranh giới câu). Cùng `chunk_size=300`, `chunk_overlap=50`, ta tạo chunk theo cả ba chiến lược để so sánh.

In [3]:
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

strategies = [ChunkStrategy.FIXED_SIZE, ChunkStrategy.RECURSIVE, ChunkStrategy.SEMANTIC]
results = {}

for strategy in strategies:
    chunker = TextChunker(strategy=strategy, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = chunker.chunk(document)
    results[strategy] = chunks
    lengths = [len(c.content) for c in chunks]
    avg_len = sum(lengths) / len(lengths) if lengths else 0
    print(f"{strategy.value:12s} -> {len(chunks):2d} chunks | "
          f"độ dài: min={min(lengths)} max={max(lengths)} tb={avg_len:.1f}")

fixed_size   ->  5 chunks | độ dài: min=255 max=300 tb=291.0
recursive    ->  5 chunks | độ dài: min=147 max=300 tb=254.2
semantic     ->  6 chunks | độ dài: min=136 max=301 tb=209.2


## 3. Xem chi tiết một vài chunk

In ra 3 chunk đầu tiên của chiến lược `RECURSIVE` để thấy rõ `chunk_id`, `start_index`/`end_index`, và phần nội dung — đồng thời quan sát phần **chồng lấp (overlap)** giữa hai chunk liên tiếp.

In [4]:
sample_chunks = results[ChunkStrategy.RECURSIVE][:3]

for chunk in sample_chunks:
    print(f"{chunk.chunk_id}  [{chunk.start_index:4d}:{chunk.end_index:4d}]  ({len(chunk.content)} ký tự)")
    print(f"  {chunk.content[:120]!r}...")
    print()

if len(sample_chunks) >= 2:
    a, b = sample_chunks[0], sample_chunks[1]
    overlap_len = a.end_index - b.start_index
    print(f"Chồng lấp giữa chunk 0 và chunk 1: {overlap_len} ký tự "
          f"(cấu hình chunk_overlap={CHUNK_OVERLAP})")

4c69f82b6454088f_chunk_0000  [   0: 147]  (147 ký tự)
  'Retrieval-Augmented Generation (RAG) la mot kien truc ket hop giua he thong\ntruy xuat thong tin (retrieval) va mo hinh s'...

4c69f82b6454088f_chunk_0001  [ 147: 425]  (278 ký tự)
  'Thay vi\nchi dua vao kien thuc da hoc trong qua trinh huan luyen, mot he thong RAG se\ntim kiem cac doan van ban lien quan'...

4c69f82b6454088f_chunk_0002  [ 425: 673]  (248 ký tự)
  'Quy trinh RAG co ban gom hai giai doan chinh: indexing (tai tai lieu, chia nho\nthanh chunk, tao embedding va luu vao vec'...

Chồng lấp giữa chunk 0 và chunk 1: 0 ký tự (cấu hình chunk_overlap=50)


## 4. Xác minh Property 1 & Property 2

- **Property 1 (bao phủ nội dung):** hợp các khoảng `[start_index:end_index)` của mọi chunk phải phủ kín mọi vị trí ký tự trong `document.content` — không bỏ sót vị trí nào.
- **Property 2 (doc_id khớp nguồn):** mọi `chunk.doc_id` phải bằng đúng `document.doc_id`.

Cell dưới kiểm tra cả hai property cho **cả ba chiến lược** bằng `assert` — nếu có vi phạm, notebook sẽ dừng lại với lỗi rõ ràng ngay tại chỗ.

In [5]:
content_length = len(document.content)

for strategy, chunks in results.items():
    # Property 2: doc_id khớp nguồn
    assert all(c.doc_id == document.doc_id for c in chunks), (
        f"[{strategy.value}] có chunk với doc_id khác document.doc_id!"
    )

    # Property 1: hợp các khoảng [start:end) phủ kín [0, content_length)
    covered = bytearray(content_length)
    for c in chunks:
        for pos in range(c.start_index, c.end_index):
            covered[pos] = 1
    fully_covered = all(covered)
    assert fully_covered, f"[{strategy.value}] còn vị trí ký tự chưa được chunk nào bao phủ!"

    print(f"{strategy.value:12s} -> Property 1 (bao phủ): OK | Property 2 (doc_id): OK")

print("\nTất cả chiến lược đều thoả Property 1 và Property 2.")

fixed_size   -> Property 1 (bao phủ): OK | Property 2 (doc_id): OK
recursive    -> Property 1 (bao phủ): OK | Property 2 (doc_id): OK
semantic     -> Property 1 (bao phủ): OK | Property 2 (doc_id): OK

Tất cả chiến lược đều thoả Property 1 và Property 2.


## 5. Ảnh hưởng của `chunk_size` lên số lượng chunk

Thử nghiệm nhanh: giữ `chunk_overlap=50`, thay đổi `chunk_size` để quan sát đánh đổi — `chunk_size` nhỏ tạo nhiều chunk hơn (ngữ cảnh mỗi chunk hẹp hơn nhưng truy hồi chính xác hơn), `chunk_size` lớn tạo ít chunk hơn (ngữ cảnh rộng hơn nhưng có thể lẫn nhiều ý không liên quan tới câu hỏi). Đây chính là tham số nên thực nghiệm với `ExperimentTracker` ở Sprint 4 để tìm cấu hình phù hợp với tài liệu thực tế.

In [6]:
for size in (150, 300, 600):
    chunker = TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=size, chunk_overlap=50)
    chunks = chunker.chunk(document)
    print(f"chunk_size={size:4d} -> {len(chunks):2d} chunks")

chunk_size= 150 ->  9 chunks
chunk_size= 300 ->  5 chunks
chunk_size= 600 ->  3 chunks


## 6. Trường hợp biên: tài liệu rỗng

Theo Yêu cầu 2.5, `chunk()` phải trả về danh sách rỗng khi `document.content` rỗng — không ném lỗi, không tạo chunk "ma".

In [7]:
from src.models import Document, DocumentType

empty_doc = Document(
    doc_id="empty_doc_demo",
    content="",
    metadata={},
    doc_type=DocumentType.TXT,
    file_path="<demo>",
)

for strategy in strategies:
    chunker = TextChunker(strategy=strategy, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = chunker.chunk(empty_doc)
    assert chunks == [], f"[{strategy.value}] tài liệu rỗng phải trả về [] chứ không phải {chunks!r}"
    print(f"{strategy.value:12s} + tài liệu rỗng -> {chunks!r} (đúng như kỳ vọng)")

fixed_size   + tài liệu rỗng -> [] (đúng như kỳ vọng)
recursive    + tài liệu rỗng -> [] (đúng như kỳ vọng)
semantic     + tài liệu rỗng -> [] (đúng như kỳ vọng)


## 7. Tổng kết

- `TextChunker` hỗ trợ ba chiến lược (`FIXED_SIZE`, `RECURSIVE`, `SEMANTIC`) với cùng tham số `chunk_size`/`chunk_overlap` — mỗi chiến lược cho số lượng và đặc điểm chunk khác nhau tuỳ vào cấu trúc tài liệu nguồn.
- **Property 1** (bao phủ trọn vẹn nội dung gốc) và **Property 2** (`chunk.doc_id == document.doc_id`) đều được xác minh trực tiếp bằng `assert` cho cả ba chiến lược — đúng như các correctness property ở design.md Phần 3.
- `chunk_size` càng nhỏ thì số chunk càng nhiều — đây là tham số cốt lõi cần thực nghiệm (sẽ dùng `ExperimentTracker` ở Sprint 4) để cân bằng giữa độ chính xác truy hồi và độ rộng ngữ cảnh.
- Trường hợp biên "tài liệu rỗng" được xử lý đúng theo Yêu cầu 2.5 — trả về `[]` mà không ném lỗi, giữ cho `RAGPipeline.index_document()` luôn ổn định.
- Bước tiếp theo trong luồng `index_document()`: mỗi `chunk.content` sẽ được `OllamaEmbeddingModel.embed_text()` chuyển thành vector — chủ đề của notebook `model_engineer/03_embedding_models.ipynb`.